# IT 7075C — Colab evidence notebook (§3.4, §3.5, §3.8)

**This notebook only runs in Google Colab.** `google.colab` does not exist in a local
kernel, so running it in VS Code fails at the first cell with
`ModuleNotFoundError: No module named 'google'`. Upload it to
[colab.research.google.com](https://colab.research.google.com) and run it there.


## §3.4 — Mount Drive and write a file

In [ ]:
# Fail with a readable message instead of a traceback if this is not Colab.
try:
    from google.colab import drive
except ModuleNotFoundError:
    raise SystemExit(
        "Not running in Google Colab. `google.colab` exists only on a Colab runtime — "
        "upload this notebook to colab.research.google.com and run it there."
    )

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import datetime

p = Path('/content/drive/MyDrive/IT7075C/persistence.txt')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(f'written before restart at {datetime.datetime.now()}\n')

print('wrote:', p)
print('exists on Drive:', p.exists())
print(p.read_text())

### ⟵ Restart the runtime now

**Runtime ▸ Restart session**, then run the next cell. Do not re-run the two cells above —
the point of the evidence is that the file was written *before* the restart.

In [ ]:
# A restart gives a fresh VM: the local disk is wiped, Drive is not.
import os

print('/content after restart (Drive not yet mounted):')
print(sorted(os.listdir('/content')))

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
p = Path('/content/drive/MyDrive/IT7075C/persistence.txt')
print()
print('survived restart:', p.exists())
print(p.read_text())

## §3.5 — Model call with the key from Colab Secrets

In [ ]:
# Install in its own cell, before importing. Colab ships an older httpx; --upgrade
# resolves it in one step so the import below picks up a consistent dependency set.
!pip install -q --upgrade anthropic
!pip show anthropic | head -2


In [ ]:
# The key lives in Colab's Secrets store, against your Google account — never in the
# notebook. This is the Colab analogue of a local .env plus .gitignore.
import os
from google.colab import userdata

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception as e:
    raise SystemExit(
        f"Could not read the secret ({type(e).__name__}). In the left sidebar open "
        "🔑 Secrets, add a secret named exactly ANTHROPIC_API_KEY, and turn on "
        "'Notebook access' for this notebook."
    )

print('ANTHROPIC_API_KEY loaded into the environment (value not displayed).')

In [ ]:
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment — no key in the source
msg = client.messages.create(
    model='claude-opus-5',
    max_tokens=100,
    messages=[{'role': 'user', 'content': 'Reply with one short sentence confirming the API call worked.'}],
)
print(f"[Anthropic claude-opus-5] {msg.content[0].text}")

## §3.8 — Resource inventory on the Colab runtime

In [ ]:
# System utility. On a CPU-only runtime nvidia-smi is absent — that is a correct
# result, not a failure. Runtime ▸ Change runtime type requests a GPU.
!nvidia-smi || echo 'no NVIDIA GPU attached (Runtime > Change runtime type to request one)'
!echo && lscpu | head -15
!echo && free -h

In [ ]:
# Framework query
import os
import torch

print('CPUs (os.cpu_count):', os.cpu_count())
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('No CUDA device — CPU-only runtime.')